In [ ]:
pip install tensorflow opencv-python

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install fvcore


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=2b3bbb4ba47a71baa41fe8ceed4670448d80360db8d97acbc4931574d14fe0bf
  Stored in directory: /root/.cache/pip/wheels/65/71/95/3b8fde5c65c6e4a806e0867c1651dcc71a1cb2f3430e8f355f
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=3eef725dd7d81bfe67fb2db9a56f615265dd7554c61def164045b5737a92f123
  Stored in directory: /root/.cache/pip/wheels/ba/5e/16/6117f8fe7e9c0c161a795e10d94645ebcf301ccbd01f66d8ec
Successfully built fvcore iopath


# CORRECT ENSEMBLE KD with Transfer Learning

In [ ]:
#!/usr/bin/env python3
# === Strict TL + KD + Pruning (filenames unchanged) ===
# - Generic TL: full fine-tuning on feline, then full fine-tuning on human
# - No head-only phases (removes odd behavior & risk of leakage via mis-freezing)
# - Strong leakage guards: fixed splits, no aug on val/test, strict eval()
# - Regularization: label smoothing, weight decay, grad clipping, early stopping
# - Pruning: global L1 prune on Conv/Linear, brief recovery finetune
# - Filenames & SAVE_ROOT kept EXACTLY as your original (so quantization snippets still work)
#
# Prints: each teacher's test metrics, ensemble test metrics, and KD student test metrics

import os, time, json, math, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

# -------- Optional deps (kept) --------
try:
    from fvcore.nn import FlopCountAnalysis
    FVCORE_OK = True
except Exception:
    FVCORE_OK = False

try:
    import onnx
    import onnxruntime as ort
    from onnxruntime.quantization import quantize_dynamic, QuantType
    ORT_OK = True
except Exception:
    ORT_OK = False

# ====================== CONFIG ======================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU = torch.device("cpu")

FELINE_IMG_PATH = "/kaggle/input/cat-reticulocyte/felineAlldata"
HUMAN_IMG_PATH  = "/kaggle/input/human-reticulocyte/SubsetAlldata"

SAVE_ROOT = Path("outputs") / "kd_transfer_onnx_table9"  # <== unchanged
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# training schedule (human student epochs kept as-is)
EPOCHS_FELINE_FULL   = 6   # feline full FT (replaces frozen+unfrozen)
EPOCHS_HUMAN_FULL    = 18  # human full FT (replaces frozen+unfrozen)
EPOCHS_STUDENT       = 20

BATCH_SIZE = 32
LR_FELINE_FULL   = 5e-5
LR_HUMAN_FULL    = 5e-5
WEIGHT_DECAY     = 1e-4
ALPHA            = 0.5
TEMPERATURE      = 4.0
GRAD_CLIP_NORM   = 2.0

NUM_WORKERS = 2
IMG_SIZE    = 224
VAL_FRAC    = 0.2
TEST_FRAC   = 0.2
FELINE_VAL_FRAC = 0.2

# Benchmark settings (kept)
FAST_BENCH = True
if FAST_BENCH:
    WARMUP_ITERS, BENCH_ITERS, BENCH_BATCH = 2, 5, 8
else:
    WARMUP_ITERS, BENCH_ITERS, BENCH_BATCH = 10, 40, 32

# ONNX/ORT providers (kept)
ORT_PROVIDERS = ["CPUExecutionProvider"]

# ====================== DATA ======================
mean_imnet = [0.485, 0.456, 0.406]
std_imnet  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])

# Build datasets without leaking transforms into the split logic
feline_base = datasets.ImageFolder(FELINE_IMG_PATH, transform=None)
human_base  = datasets.ImageFolder(HUMAN_IMG_PATH, transform=None)
assert len(feline_base.classes) >= 2, "Feline dataset must have >=2 classes."
NUM_CLASSES = len(human_base.classes)

def stratified_split_indices_by_frac(base_ds, val_frac, test_frac=0.0, seed=SEED):
    """Split on base_ds.samples (paths+targets) to avoid transform side-effects; stratified & seeded."""
    rng = np.random.default_rng(seed)
    targets = np.array([y for _, y in base_ds.samples])
    idx = np.arange(len(targets))
    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(targets):
        ci = idx[targets==c].copy(); rng.shuffle(ci)
        n = len(ci)
        n_val  = max(1, int(round(n*val_frac)))
        n_test = max(0, int(round(n*test_frac)))
        n_val = min(n_val, n-1) if n>1 else 1
        val_idx.extend(ci[:n_val])
        test_idx.extend(ci[n_val:n_val+n_test])
        train_idx.extend(ci[n_val+n_test:])
    return train_idx, val_idx, test_idx

# feline split
f_tr_idx, f_va_idx, _ = stratified_split_indices_by_frac(feline_base, FELINE_VAL_FRAC, 0.0)
feline_train = Subset(datasets.ImageFolder(FELINE_IMG_PATH, transform=train_tf), f_tr_idx)
feline_val   = Subset(datasets.ImageFolder(FELINE_IMG_PATH, transform=eval_tf),  f_va_idx)

# human split
h_tr_idx, h_va_idx, h_te_idx = stratified_split_indices_by_frac(human_base, VAL_FRAC, TEST_FRAC)
human_train = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=train_tf), h_tr_idx)
human_val   = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  h_va_idx)
human_test  = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  h_te_idx)

print("Feline:", len(feline_train), "train,", len(feline_val), "val", datasets.ImageFolder(FELINE_IMG_PATH).classes)
print("Human :", len(human_train), "train,", len(human_val), "val,", len(human_test), "test", datasets.ImageFolder(HUMAN_IMG_PATH).classes)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True)

feline_train_loader = make_loader(feline_train, shuffle=True)
feline_val_loader   = make_loader(feline_val,   shuffle=False)
human_train_loader  = make_loader(human_train,  shuffle=True)
human_val_loader    = make_loader(human_val,    shuffle=False)
human_test_loader   = make_loader(human_test,   shuffle=False)

# ====================== UTILS ======================
def replace_classifier_for_num_classes(model, num_classes):
    if isinstance(model, models.SqueezeNet):
        model.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=(1,1))
        model.num_classes = num_classes
    elif hasattr(model, 'classifier') and isinstance(model.classifier, nn.Sequential):
        in_features = None
        for layer in reversed(model.classifier):
            if isinstance(layer, nn.Linear):
                in_features = layer.in_features; break
        if in_features is None and hasattr(model, 'fc'):
            in_features = model.fc.in_features
            model.fc = nn.Linear(in_features, num_classes)
        else:
            model.classifier[-1] = nn.Linear(in_features, num_classes)
    elif hasattr(model, 'fc'):
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
    else:
        raise ValueError("Unsupported model type.")
    return model

def count_params_m(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def compute_flops_g(model, input_size=(1,3,IMG_SIZE,IMG_SIZE)):
    if not FVCORE_OK: return float("nan")
    m = model.to(CPU).eval()
    x = torch.randn(*input_size)
    try:
        return float(FlopCountAnalysis(m, x).total() / 1e9)
    except Exception:
        return float("nan")

def save_state(model, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), path)
    return path.stat().st_size / (1024*1024)

def optim_for(params, lr):
    return torch.optim.Adam(params, lr=lr, weight_decay=WEIGHT_DECAY)

def run_epoch(model, loader, loss_fn, opt=None, device=DEVICE, grad_clip=None):
    train = opt is not None
    model.train(train)
    total, correct, n = 0.0, 0, 0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        if train: opt.zero_grad(set_to_none=True)
        out = model(x)
        loss = loss_fn(out, y)
        if train:
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
        total += loss.item()*x.size(0)
        correct += (out.argmax(1)==y).sum().item()
        n += x.size(0)
    return total/max(n,1), correct/max(n,1)

@torch.no_grad()
def eval_metrics(model, loader, device=DEVICE):
    model.eval()
    y_true, y_pred = [], []
    for x,y in loader:
        x = x.to(device)
        y_true.extend(y.numpy().tolist())
        y_pred.extend(model(x).argmax(1).cpu().numpy().tolist())
    acc = accuracy_score(y_true, y_pred)*100.0
    f1m = f1_score(y_true, y_pred, average="macro")*100.0
    return acc, f1m

class EarlyStopper:
    def __init__(self, patience=5, mode="max", delta=0.0):
        self.patience, self.mode, self.delta = patience, mode, delta
        self.best = -float("inf") if mode=="max" else float("inf")
        self.count = 0
    def step(self, metric):
        improve = (metric > self.best + self.delta) if self.mode=="max" else (metric < self.best - self.delta)
        if improve:
            self.best = metric; self.count = 0; return True
        else:
            self.count += 1; return False
    def should_stop(self): return self.count >= self.patience

# ====================== Pruning helpers ======================
import torch.nn.utils.prune as prune

def modules_to_prune(model):
    pairs = []
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            pairs.append((m, 'weight'))
    return pairs

def apply_global_prune(model, amount=0.2):
    params = modules_to_prune(model)
    if len(params)==0: return
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=amount)
    # Convert pruned weights into real tensors (remove masks & reparam)
    for m, name in params:
        try:
            prune.remove(m, name)
        except Exception:
            pass

# ====================== Generic TL Teacher ======================
def pretrain_teacher(arch_name, model_fn):
    out_dir = SAVE_ROOT / f"teacher_{arch_name}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Feline full fine-tune
    model = model_fn(weights="DEFAULT" if "weights" in model_fn.__code__.co_varnames else None)
    model = replace_classifier_for_num_classes(model, len(datasets.ImageFolder(FELINE_IMG_PATH).classes)).to(DEVICE)
    ce_feline = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt = optim_for(model.parameters(), LR_FELINE_FULL)
    es_feline = EarlyStopper(patience=3, mode="max")
    best_feline_path = out_dir / f"{arch_name}_feline_best.pth"  # <== unchanged filename
    best_feline_acc = -1.0

    for ep in range(1, EPOCHS_FELINE_FULL+1):
        tr_loss, tr_acc = run_epoch(model, feline_train_loader, ce_feline, opt, DEVICE, grad_clip=GRAD_CLIP_NORM)
        v_acc, v_f1 = eval_metrics(model, feline_val_loader, DEVICE)
        print(f"[{arch_name}][Feline] {ep}/{EPOCHS_FELINE_FULL} loss={tr_loss:.6f} val_acc={v_acc:.2f} f1={v_f1:.2f}")
        if es_feline.step(v_acc):
            save_state(model, best_feline_path); best_feline_acc = v_acc
        if es_feline.should_stop(): break

    model.load_state_dict(torch.load(best_feline_path, map_location=DEVICE))

    # 2) Switch head for human & full fine-tune
    model = replace_classifier_for_num_classes(model, NUM_CLASSES).to(DEVICE)
    ce_human = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt = optim_for(model.parameters(), LR_HUMAN_FULL)
    es_human = EarlyStopper(patience=4, mode="max")
    best_human_path = out_dir / f"{arch_name}_human_best.pth"  # <== unchanged filename
    best_human_acc = -1.0

    for ep in range(1, EPOCHS_HUMAN_FULL+1):
        tr_loss, tr_acc = run_epoch(model, human_train_loader, ce_human, opt, DEVICE, grad_clip=GRAD_CLIP_NORM)
        v_acc, v_f1 = eval_metrics(model, human_val_loader, DEVICE)
        print(f"[{arch_name}][Human] {ep}/{EPOCHS_HUMAN_FULL} loss={tr_loss:.6f} val_acc={v_acc:.2f} f1={v_f1:.2f}")
        if es_human.step(v_acc):
            save_state(model, best_human_path); best_human_acc = v_acc
        if es_human.should_stop(): break

    model.load_state_dict(torch.load(best_human_path, map_location=DEVICE))

    # 3) Prune + short recovery finetune on human train (strictness against overfit)
    apply_global_prune(model, amount=0.2)  # 20% global L1
    opt = optim_for(model.parameters(), LR_HUMAN_FULL/2)
    es_rec = EarlyStopper(patience=3, mode="max")
    for ep in range(1, 6):  # brief recovery
        tr_loss, tr_acc = run_epoch(model, human_train_loader, ce_human, opt, DEVICE, grad_clip=GRAD_CLIP_NORM)
        v_acc, v_f1 = eval_metrics(model, human_val_loader, DEVICE)
        print(f"[{arch_name}][Prune-Recover] {ep}/5 loss={tr_loss:.6f} val_acc={v_acc:.2f} f1={v_f1:.2f}")
        if es_rec.step(v_acc):
            save_state(model, best_human_path); best_human_acc = v_acc
        if es_rec.should_stop(): break

    model.load_state_dict(torch.load(best_human_path, map_location=DEVICE))
    return model, out_dir

# ====================== Train Teachers ======================
teachers_cfg = [
    ("mobilenet_v2",    models.mobilenet_v2),
    ("efficientnet_b0", models.efficientnet_b0),
    ("squeezenet1_1",   models.squeezenet1_1),
]
teachers = {}
for name, fn in teachers_cfg:
    print(f"\n==== Train Teacher: {name} ====")
    model, out_dir = pretrain_teacher(name, fn)
    acc, f1m = eval_metrics(model, human_test_loader, DEVICE)
    teachers[name] = {"model": model, "dir": out_dir, "test_acc": acc, "test_f1m": f1m}
    print(f"[{name}] TEST  acc={acc:.2f}  f1={f1m:.2f}")

# ====================== Ensemble Eval (soft voting) ======================
@torch.no_grad()
def ensemble_eval(teachers, loader, device=DEVICE):
    models_list = [t["model"].to(device).eval() for t in teachers.values()]
    y_true, y_pred = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        probs = None
        for m in models_list:
            p = F.softmax(m(imgs), dim=1)
            probs = p if probs is None else probs + p
        preds = (probs/len(models_list)).argmax(1).cpu().numpy().tolist()
        y_pred.extend(preds); y_true.extend(labels.numpy().tolist())
    acc = accuracy_score(y_true, y_pred)*100.0
    f1m = f1_score(y_true, y_pred, average="macro")*100.0
    return acc, f1m

ens_acc, ens_f1m = ensemble_eval(teachers, human_test_loader, DEVICE)
print(f"\n[Ensemble] TEST acc={ens_acc:.2f}  f1={ens_f1m:.2f}")

# ====================== KD Student ======================
class StudentNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = models.shufflenet_v2_x0_5(weights="DEFAULT")
        in_f = base.fc.in_features
        # add mild dropout for regularization
        base.fc = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(in_f, num_classes))
        self.model = base
    def forward(self, x): return self.model(x)

student = StudentNet(NUM_CLASSES).to(DEVICE)
opt_s = torch.optim.Adam(student.parameters(), lr=1e-4, weight_decay=WEIGHT_DECAY)
ce_s  = nn.CrossEntropyLoss(label_smoothing=0.05)
kl = nn.KLDivLoss(reduction="batchmean")
t_models = [t["model"].to(DEVICE).eval() for t in teachers.values()]

def kd_epoch(student, loader):
    student.train()
    total, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            ps = [F.softmax(m(imgs)/TEMPERATURE, dim=1) for m in t_models]
            t_soft = sum(ps)/len(ps)
        logits = student(imgs)
        loss = ALPHA*ce_s(logits, labels) + (1-ALPHA)*kl(F.log_softmax(logits/TEMPERATURE, dim=1), t_soft)
        opt_s.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP_NORM)
        opt_s.step()
        total += loss.item()*imgs.size(0)
        correct += (logits.argmax(1)==labels).sum().item(); n += imgs.size(0)
    return total/max(n,1), correct/max(n,1)

best_v, best_path = -1.0, SAVE_ROOT / "student_best.pth"  # <== unchanged filename
es_stud = EarlyStopper(patience=5, mode="max")

for ep in range(1, EPOCHS_STUDENT+1):
    tr_loss, tr_acc = kd_epoch(student, human_train_loader)
    v_acc, v_f1 = eval_metrics(student, human_val_loader, DEVICE)
    print(f"[Student KD] {ep}/{EPOCHS_STUDENT} loss={tr_loss:.6f} train_acc={tr_acc*100:.2f} val_acc={v_acc:.2f} f1={v_f1:.2f}")
    if es_stud.step(v_acc):
        save_state(student, best_path); best_v = v_acc
    if es_stud.should_stop(): break

student.load_state_dict(torch.load(best_path, map_location=DEVICE))

# Optional: light prune student & recover briefly (tighten generalization), then re-save SAME filename
apply_global_prune(student, amount=0.15)
opt_rec = torch.optim.Adam(student.parameters(), lr=7.5e-5, weight_decay=WEIGHT_DECAY)
es_re = EarlyStopper(patience=3, mode="max")
for ep in range(1, 5):
    tr_loss, tr_acc = kd_epoch(student, human_train_loader)
    v_acc, v_f1 = eval_metrics(student, human_val_loader, DEVICE)
    print(f"[Student Prune-Recover] {ep}/4 loss={tr_loss:.6f} val_acc={v_acc:.2f} f1={v_f1:.2f}")
    if es_re.step(v_acc): save_state(student, best_path)
    if es_re.should_stop(): break
student.load_state_dict(torch.load(best_path, map_location=DEVICE))

# Final strict evaluations
for name, t in teachers.items():
    acc, f1m = eval_metrics(t["model"], human_test_loader, DEVICE)
    print(f"[{name}] FINAL TEST acc={acc:.2f} f1={f1m:.2f}")
ens_acc, ens_f1m = ensemble_eval(teachers, human_test_loader, DEVICE)
print(f"[Ensemble] FINAL TEST acc={ens_acc:.2f} f1={ens_f1m:.2f}")

stud_acc, stud_f1 = eval_metrics(student, human_test_loader, DEVICE)
print(f"[Student] FINAL TEST acc={stud_acc:.2f} f1={stud_f1:.2f}")


Feline: 2354 train, 588 val ['erythrocyte', 'reticulocyte']
Human : 872 train, 291 val, 291 test ['BG', 'erythrocyte', 'reticulocyte']

==== Train Teacher: mobilenet_v2 ====
[mobilenet_v2][Feline] 1/6 loss=0.660858 val_acc=67.35 f1=40.24
[mobilenet_v2][Feline] 2/6 loss=0.643127 val_acc=67.35 f1=40.24
[mobilenet_v2][Feline] 3/6 loss=0.641209 val_acc=67.35 f1=40.24
[mobilenet_v2][Feline] 4/6 loss=0.634682 val_acc=70.75 f1=52.25
[mobilenet_v2][Feline] 5/6 loss=0.632045 val_acc=71.09 f1=53.44
[mobilenet_v2][Feline] 6/6 loss=0.623056 val_acc=70.41 f1=49.57
[mobilenet_v2][Human] 1/18 loss=1.079108 val_acc=48.45 f1=44.18
[mobilenet_v2][Human] 2/18 loss=1.036894 val_acc=65.29 f1=65.04
[mobilenet_v2][Human] 3/18 loss=0.971017 val_acc=76.63 f1=76.79
[mobilenet_v2][Human] 4/18 loss=0.890965 val_acc=75.60 f1=75.83
[mobilenet_v2][Human] 5/18 loss=0.871084 val_acc=80.41 f1=80.47
[mobilenet_v2][Human] 6/18 loss=0.815309 val_acc=80.41 f1=80.79
[mobilenet_v2][Human] 7/18 loss=0.817969 val_acc=79.38 f1=

Downloading: "https://download.pytorch.org/models/shufflenetv2_x0.5-f707e7126e.pth" to /root/.cache/torch/hub/checkpoints/shufflenetv2_x0.5-f707e7126e.pth



[Ensemble] TEST acc=92.10  f1=92.21


100%|██████████| 5.28M/5.28M [00:00<00:00, 135MB/s]


[Student KD] 1/20 loss=0.566771 train_acc=51.03 val_acc=76.98 f1=75.17
[Student KD] 2/20 loss=0.556920 train_acc=67.89 val_acc=82.47 f1=81.22
[Student KD] 3/20 loss=0.538418 train_acc=79.36 val_acc=87.29 f1=86.87
[Student KD] 4/20 loss=0.504507 train_acc=80.05 val_acc=86.94 f1=86.59
[Student KD] 5/20 loss=0.452437 train_acc=82.68 val_acc=87.63 f1=87.47
[Student KD] 6/20 loss=0.388724 train_acc=84.06 val_acc=89.35 f1=89.16
[Student KD] 7/20 loss=0.333808 train_acc=84.06 val_acc=90.03 f1=89.88
[Student KD] 8/20 loss=0.293184 train_acc=86.47 val_acc=91.75 f1=91.61
[Student KD] 9/20 loss=0.258688 train_acc=88.53 val_acc=94.50 f1=94.42
[Student KD] 10/20 loss=0.245701 train_acc=89.22 val_acc=94.85 f1=94.82
[Student KD] 11/20 loss=0.236691 train_acc=88.53 val_acc=97.59 f1=97.57
[Student KD] 12/20 loss=0.221390 train_acc=89.56 val_acc=97.59 f1=97.58
[Student KD] 13/20 loss=0.214542 train_acc=90.25 val_acc=96.91 f1=96.85
[Student KD] 14/20 loss=0.188830 train_acc=93.81 val_acc=97.59 f1=97.57
[

In [ ]:
!pip install -U onnx onnxruntime onnxruntime-tools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 77.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.7/212.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-api-core 1.34.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.2

In [ ]:
import onnxruntime as ort
print("ORT:", ort.__version__)
print("Providers:", ort.get_available_providers())
# should include: ['CPUExecutionProvider']


ORT: 1.22.1
Providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


In [ ]:
!pip uninstall -y onnxruntime || true
!pip install -U onnx onnxruntime-gpu onnxruntime-tools


Found existing installation: onnxruntime 1.22.1
Uninstalling onnxruntime-1.22.1:
  Successfully uninstalled onnxruntime-1.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 5.9 MB/s eta 0:00:00:00:0100:01


In [ ]:
import onnxruntime as ort
print("ORT:", ort.__version__)
print("Providers:", ort.get_available_providers())
# expect: ['CUDAExecutionProvider', 'CPUExecutionProvider']  (TensorRT may not be available on Kaggle)


ORT: 1.22.1
Providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


# New Attempt

In [ ]:
%%writefile onnx_quant_bench.py
# (paste the full script content here)


Writing onnx_quant_bench.py


In [ ]:
!ls -lah | grep onnx_quant_bench.py


-rw-r--r-- 1 root root   39 Aug 13 06:22 onnx_quant_bench.py


In [ ]:
!ls -lah


total 20K
drwxr-xr-x 4 root root 4.0K Aug 13 06:22 .
drwxr-xr-x 5 root root 4.0K Aug 13 06:04 ..
-rw-r--r-- 1 root root   39 Aug 13 06:22 onnx_quant_bench.py
drwxr-xr-x 3 root root 4.0K Aug 13 06:06 outputs
drwxr-xr-x 2 root root 4.0K Aug 13 06:05 .virtual_documents


In [ ]:
!ls outputs/kd_transfer_onnx_table9


student_best.pth	 teacher_mobilenet_v2
teacher_efficientnet_b0  teacher_squeezenet1_1


# Fixed Quantized Output

In [ ]:
import torch, torch.nn as nn
from torchvision import models, datasets, transforms
from pathlib import Path

# === paths (MUST match your training) ===
SAVE_ROOT = Path("outputs") / "kd_transfer_onnx_table9"
CKPT = {
    "mobilenet_v2": SAVE_ROOT/"teacher_mobilenet_v2"/"mobilenet_v2_human_best.pth",
    "efficientnet_b0": SAVE_ROOT/"teacher_efficientnet_b0"/"efficientnet_b0_human_best.pth",
    "squeezenet1_1": SAVE_ROOT/"teacher_squeezenet1_1"/"squeezenet1_1_human_best.pth",
    "student": SAVE_ROOT/"student_best.pth",
}

# how many classes (use the human dataset you trained on)
HUMAN_IMG_PATH = "/kaggle/input/human-reticulocyte/SubsetAlldata"
num_classes = len(datasets.ImageFolder(HUMAN_IMG_PATH).classes)

# build nets exactly like training
def replace_head(m, n):
    if isinstance(m, models.SqueezeNet):
        m.classifier[1] = nn.Conv2d(512, n, kernel_size=1)
    elif hasattr(m, "classifier") and isinstance(m.classifier, nn.Sequential):
        # MobileNet/EfficientNet
        for i in range(len(m.classifier)-1, -1, -1):
            if isinstance(m.classifier[i], nn.Linear):
                in_f = m.classifier[i].in_features
                m.classifier[i] = nn.Linear(in_f, n)
                break
    elif hasattr(m, "fc"):
        in_f = m.fc.in_features
        m.fc = nn.Linear(in_f, n)
    return m

def make_model(tag):
    if tag=="mobilenet_v2":
        m = models.mobilenet_v2(weights=None)
    elif tag=="efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
    elif tag=="squeezenet1_1":
        m = models.squeezenet1_1(weights=None)
    elif tag=="student":
        base = models.shufflenet_v2_x0_5(weights=None)
        in_f = base.fc.in_features
        base.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, num_classes))
        class S(nn.Module):
            def __init__(self, b): super().__init__(); self.model=b
            def forward(self,x): return self.model(x)
        m = S(base)
        return m
    return replace_head(m, num_classes)

def export_onnx(tag, ckpt_path, out_path):
    assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"
    m = make_model(tag)
    m.load_state_dict(torch.load(ckpt_path, map_location="cpu"), strict=False)
    m.eval()
    x = torch.randn(1,3,224,224, dtype=torch.float32)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.onnx.export(
        m, x, str(out_path),
        export_params=True, do_constant_folding=True,
        input_names=["input"], output_names=["logits"],
        dynamic_axes=None, opset_version=13
    )
    print(f"[OK] Exported {tag} -> {out_path}")

# export all 4
export_onnx("mobilenet_v2", CKPT["mobilenet_v2"], SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_FP32.onnx")
export_onnx("efficientnet_b0", CKPT["efficientnet_b0"], SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_FP32.onnx")
export_onnx("squeezenet1_1", CKPT["squeezenet1_1"], SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_FP32.onnx")
export_onnx("student", CKPT["student"], SAVE_ROOT/"onnx_student"/"student_FP32.onnx")


[OK] Exported mobilenet_v2 -> outputs/kd_transfer_onnx_table9/onnx_mobilenet_v2/mobilenet_v2_FP32.onnx
[OK] Exported efficientnet_b0 -> outputs/kd_transfer_onnx_table9/onnx_efficientnet_b0/efficientnet_b0_FP32.onnx
[OK] Exported squeezenet1_1 -> outputs/kd_transfer_onnx_table9/onnx_squeezenet1_1/squeezenet1_1_FP32.onnx
[OK] Exported student -> outputs/kd_transfer_onnx_table9/onnx_student/student_FP32.onnx


In [ ]:
import torch, torch.nn as nn
from torchvision import models, datasets
from pathlib import Path

SAVE_ROOT = Path("outputs") / "kd_transfer_onnx_table9"
CKPT = {
    "mobilenet_v2": SAVE_ROOT/"teacher_mobilenet_v2"/"mobilenet_v2_human_best.pth",
    "efficientnet_b0": SAVE_ROOT/"teacher_efficientnet_b0"/"efficientnet_b0_human_best.pth",
    "squeezenet1_1": SAVE_ROOT/"teacher_squeezenet1_1"/"squeezenet1_1_human_best.pth",
    "student": SAVE_ROOT/"student_best.pth",
}

HUMAN_IMG_PATH = "/kaggle/input/human-reticulocyte/SubsetAlldata"
num_classes = len(datasets.ImageFolder(HUMAN_IMG_PATH).classes)

def replace_head(m, n):
    if isinstance(m, models.SqueezeNet):
        m.classifier[1] = nn.Conv2d(512, n, kernel_size=1)
    elif hasattr(m, "classifier") and isinstance(m.classifier, nn.Sequential):
        for i in range(len(m.classifier)-1, -1, -1):
            if isinstance(m.classifier[i], nn.Linear):
                in_f = m.classifier[i].in_features
                m.classifier[i] = nn.Linear(in_f, n); break
    elif hasattr(m, "fc"):
        in_f = m.fc.in_features; m.fc = nn.Linear(in_f, n)
    return m

def make_model(tag):
    if tag=="mobilenet_v2":
        m = models.mobilenet_v2(weights=None)
    elif tag=="efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
    elif tag=="squeezenet1_1":
        m = models.squeezenet1_1(weights=None)
    elif tag=="student":
        base = models.shufflenet_v2_x0_5(weights=None)
        in_f = base.fc.in_features
        base.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, num_classes))
        class S(nn.Module):
            def __init__(self, b): super().__init__(); self.model=b
            def forward(self,x): return self.model(x)
        return S(base)
    return replace_head(m, num_classes)

def export_onnx(tag, ckpt_path, out_path):
    m = make_model(tag)
    m.load_state_dict(torch.load(ckpt_path, map_location="cpu"), strict=False)
    m.eval()
    x = torch.randn(1,3,224,224, dtype=torch.float32)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.onnx.export(
        m, x, str(out_path),
        export_params=True, do_constant_folding=True,
        input_names=["input"], output_names=["logits"],
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},  # <-- dynamic batch
        opset_version=13
    )
    print(f"[OK] Exported {tag} -> {out_path}")

export_onnx("mobilenet_v2", CKPT["mobilenet_v2"], SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_FP32.onnx")
export_onnx("efficientnet_b0", CKPT["efficientnet_b0"], SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_FP32.onnx")
export_onnx("squeezenet1_1",   CKPT["squeezenet1_1"],   SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_FP32.onnx")
export_onnx("student",         CKPT["student"],         SAVE_ROOT/"onnx_student"/"student_FP32.onnx")


[OK] Exported mobilenet_v2 -> outputs/kd_transfer_onnx_table9/onnx_mobilenet_v2/mobilenet_v2_FP32.onnx
[OK] Exported efficientnet_b0 -> outputs/kd_transfer_onnx_table9/onnx_efficientnet_b0/efficientnet_b0_FP32.onnx
[OK] Exported squeezenet1_1 -> outputs/kd_transfer_onnx_table9/onnx_squeezenet1_1/squeezenet1_1_FP32.onnx
[OK] Exported student -> outputs/kd_transfer_onnx_table9/onnx_student/student_FP32.onnx


In [ ]:
import numpy as np
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
from torchvision import datasets, transforms

IMG_SIZE=224
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
calib_ds = datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf)

class CalibReader(CalibrationDataReader):
    def __init__(self, ds, n_batches=20, bs=8):
        self.ds=ds; self.n=n_batches; self.bs=bs; self.i=0
        idx = np.random.default_rng(123).permutation(len(ds))[:n_batches*bs]
        self.batches=[]
        for k in range(0,len(idx),bs):
            xs=[ds[j][0].numpy() for j in idx[k:k+bs]]
            self.batches.append({"input": np.stack(xs,0).astype(np.float32)})
    def get_next(self):
        if self.i>=len(self.batches): return None
        b=self.batches[self.i]; self.i+=1; return b

def qdq_int8(fp32_path, int8_path):
    int8_path.parent.mkdir(parents=True, exist_ok=True)
    quantize_static(
        model_input=str(fp32_path),
        model_output=str(int8_path),
        calibration_data_reader=CalibReader(calib_ds, n_batches=20, bs=8),
        quant_format=QuantFormat.QDQ,
        per_channel=True,
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QUInt8,
        reduce_range=False
    )
    print(f"[OK] Quantized -> {int8_path}")

qdq_int8(SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_FP32.onnx",     SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_INT8.onnx")
qdq_int8(SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_FP32.onnx", SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_INT8.onnx")
qdq_int8(SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_FP32.onnx",   SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_INT8.onnx")
qdq_int8(SAVE_ROOT/"onnx_student"/"student_FP32.onnx",               SAVE_ROOT/"onnx_student"/"student_INT8.onnx")


[OK] Quantized -> outputs/kd_transfer_onnx_table9/onnx_mobilenet_v2/mobilenet_v2_INT8.onnx
[OK] Quantized -> outputs/kd_transfer_onnx_table9/onnx_efficientnet_b0/efficientnet_b0_INT8.onnx
[OK] Quantized -> outputs/kd_transfer_onnx_table9/onnx_squeezenet1_1/squeezenet1_1_INT8.onnx
[OK] Quantized -> outputs/kd_transfer_onnx_table9/onnx_student/student_INT8.onnx


In [ ]:
import onnxruntime as ort, numpy as np, time
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader

def mk_sess(p):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    so.intra_op_num_threads = 4; so.inter_op_num_threads = 1
    return ort.InferenceSession(str(p), sess_options=so, providers=["CPUExecutionProvider"])

def acc_f1(sess, loader):
    nm = sess.get_inputs()[0].name
    y_true, y_pred = [], []
    for x,y in loader:
        arr = x.numpy().astype(np.float32)
        logits = sess.run(None, {nm: arr})[0]
        y_pred.extend(np.argmax(logits,1).tolist())
        y_true.extend(y.numpy().tolist())
    return accuracy_score(y_true,y_pred)*100, f1_score(y_true,y_pred,average="macro")*100

def bench(sess, batch=(8,3,224,224), iters=30):
    nm = sess.get_inputs()[0].name
    x = np.random.randn(*batch).astype(np.float32)
    for _ in range(5): sess.run(None,{nm:x})
    t0=time.perf_counter()
    for _ in range(iters): sess.run(None,{nm:x})
    t1=time.perf_counter()
    ms=(t1-t0)*1000/iters; fps=(batch[0]*iters)/(t1-t0)
    return ms,fps

eval_loader = DataLoader(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),
                         batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

pairs = [
    ("mobilenet_v2", SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_FP32.onnx", SAVE_ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_INT8.onnx"),
    ("efficientnet_b0", SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_FP32.onnx", SAVE_ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_INT8.onnx"),
    ("squeezenet1_1", SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_FP32.onnx", SAVE_ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_INT8.onnx"),
    ("student", SAVE_ROOT/"onnx_student"/"student_FP32.onnx", SAVE_ROOT/"onnx_student"/"student_INT8.onnx"),
]

for name, fp32_p, int8_p in pairs:
    s_fp32 = mk_sess(fp32_p); s_int8 = mk_sess(int8_p)
    a32,f132 = acc_f1(s_fp32, eval_loader)
    a8 ,f18  = acc_f1(s_int8, eval_loader)
    m32,fps32 = bench(s_fp32); m8,fps8 = bench(s_int8)
    print(f"\n{name}:")
    print(f"  FP32  acc={a32:.2f} f1={f132:.2f}  lat={m32:.2f}ms  thr={fps32:.1f}fps")
    print(f"  INT8  acc={a8:.2f}  f1={f18:.2f}   lat={m8:.2f}ms   thr={fps8:.1f}fps  Δacc={a8-a32:.2f}pp")


2025-08-13 06:37:48.019798764 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.17/conv/conv.1/conv.1.2/Constant_1_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:37:48.019835244 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.17/conv/conv.0/conv.0.2/Constant_1_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:37:48.019841155 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.16/conv/conv.1/conv.1.2/Constant_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:37:48.019846231 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.16/conv/conv.0/conv.0.2/Constant_output_0'. It is not used by any node and should be removed from the model.
2025-08-


mobilenet_v2:
  FP32  acc=91.40 f1=91.51  lat=37.13ms  thr=215.5fps
  INT8  acc=83.49  f1=83.24   lat=29.77ms   thr=268.8fps  Δacc=-7.91pp

efficientnet_b0:
  FP32  acc=88.72 f1=88.81  lat=130.25ms  thr=61.4fps
  INT8  acc=88.17  f1=88.43   lat=85.62ms   thr=93.4fps  Δacc=-0.55pp

squeezenet1_1:
  FP32  acc=81.98 f1=82.06  lat=21.45ms  thr=372.9fps
  INT8  acc=81.84  f1=82.02   lat=24.21ms   thr=330.4fps  Δacc=-0.14pp

student:
  FP32  acc=99.31 f1=99.31  lat=9.71ms  thr=823.8fps
  INT8  acc=89.27  f1=89.26   lat=25.23ms   thr=317.1fps  Δacc=-10.04pp


In [ ]:
# === Comprehensive FP32 vs INT8 benchmark for teachers, student, and 3x ensemble ===
# Metrics: Params (M), Disk Size (MB), Latency (ms), Throughput (fps), Accuracy (%), Macro F1 (%), Speed Up (INT8 vs FP32)
import os, time, math, numpy as np, pandas as pd, onnx, onnxruntime as ort
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ---------- Paths (match your project) ----------
ROOT = Path("outputs") / "kd_transfer_onnx_table9"
PATHS = {
    "mobilenet_v2": {
        "fp32": ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_FP32.onnx",
        "int8": ROOT/"onnx_mobilenet_v2"/"mobilenet_v2_INT8.onnx",
    },
    "efficientnet_b0": {
        "fp32": ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_FP32.onnx",
        "int8": ROOT/"onnx_efficientnet_b0"/"efficientnet_b0_INT8.onnx",
    },
    "squeezenet1_1": {
        "fp32": ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_FP32.onnx",
        "int8": ROOT/"onnx_squeezenet1_1"/"squeezenet1_1_INT8.onnx",
    },
    "student": {
        "fp32": ROOT/"onnx_student"/"student_FP32.onnx",
        "int8": ROOT/"onnx_student"/"student_INT8.onnx",
    },
}

# ---------- Data (same preprocessing as training) ----------
IMG_SIZE = 224
HUMAN_IMG_PATH = "/kaggle/input/human-reticulocyte/SubsetAlldata"
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_ds = datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf)
eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# ---------- ORT session helper ----------
def mk_sess(path, intra=4, inter=1):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    so.intra_op_num_threads = intra
    so.inter_op_num_threads = inter
    so.enable_mem_pattern = True
    so.enable_cpu_mem_arena = True
    return ort.InferenceSession(str(path), sess_options=so, providers=["CPUExecutionProvider"])

# ---------- Metrics helpers ----------
def onnx_params_m_and_size_mb(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing ONNX: {path}")
    model = onnx.load(str(path))
    params = sum(int(np.prod(i.dims)) for i in model.graph.initializer) / 1e6  # Millions
    size_mb = path.stat().st_size / (1024*1024)
    return params, size_mb

def eval_acc_f1(sess, loader):
    nm = sess.get_inputs()[0].name
    # Light warmup
    for _ in range(3):
        x,_ = next(iter(loader))
        arr = x.numpy().astype(np.float32)
        sess.run(None, {nm: arr})
    y_true, y_pred = [], []
    for x,y in loader:
        arr = x.numpy().astype(np.float32)
        logits = sess.run(None, {nm: arr})[0]
        y_pred.extend(np.argmax(logits,1).tolist())
        y_true.extend(y.numpy().tolist())
    return accuracy_score(y_true,y_pred)*100.0, f1_score(y_true,y_pred,average="macro")*100.0

def bench(sess, batch=(8,3,IMG_SIZE,IMG_SIZE), iters=50, warmup=10):
    nm = sess.get_inputs()[0].name
    x = np.random.randn(*batch).astype(np.float32)
    for _ in range(warmup): sess.run(None,{nm:x})
    t0 = time.perf_counter()
    for _ in range(iters): sess.run(None,{nm:x})
    t1 = time.perf_counter()
    ms  = (t1-t0)*1000/iters
    fps = (batch[0]*iters)/(t1-t0)
    return ms, fps

def softmax(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x); return e/np.sum(e, axis=axis, keepdims=True)

def ensemble_eval(sessions, loader):
    names = [s.get_inputs()[0].name for s in sessions]
    # warmup
    for _ in range(3):
        x,_ = next(iter(loader))
        arr = x.numpy().astype(np.float32)
        for s,nm in zip(sessions, names): s.run(None, {nm: arr})
    y_true, y_pred = [], []
    for x,y in loader:
        arr = x.numpy().astype(np.float32)
        probs=None
        for s,nm in zip(sessions, names):
            logits = s.run(None, {nm: arr})[0]
            p = softmax(logits, axis=1)
            probs = p if probs is None else probs+p
        preds = np.argmax(probs/len(sessions),1)
        y_pred.extend(preds.tolist()); y_true.extend(y.numpy().tolist())
    return accuracy_score(y_true,y_pred)*100.0, f1_score(y_true,y_pred,average="macro")*100.0

def ensemble_bench(sessions, batch=(8,3,IMG_SIZE,IMG_SIZE), iters=50, warmup=10):
    names = [s.get_inputs()[0].name for s in sessions]
    x = np.random.randn(*batch).astype(np.float32)
    for _ in range(warmup):
        for s,nm in zip(sessions,names): s.run(None,{nm:x})
    t0 = time.perf_counter()
    for _ in range(iters):
        for s,nm in zip(sessions,names): s.run(None,{nm:x})
    t1 = time.perf_counter()
    ms  = (t1-t0)*1000/iters
    fps = (batch[0]*iters*len(sessions))/(t1-t0)
    return ms, fps

# ---------- Build the comprehensive table ----------
rows = []
def add_model_rows(name, fp32_path, int8_path):
    # FP32
    p_m, size32 = onnx_params_m_and_size_mb(fp32_path)
    s32 = mk_sess(fp32_path)
    acc32, f132 = eval_acc_f1(s32, eval_loader)
    ms32, fps32 = bench(s32)

    rows.append({
        "Group": name, "Variant": "FP32",
        "Parameters (M)": round(p_m,3), "Disk Size (MB)": round(size32,2),
        "Inference (ms)": round(ms32,2), "Throughput (fps)": round(fps32,1),
        "Accuracy (%)": round(acc32,2), "Macro F1 (%)": round(f132,2),
        "Speed Up": 1.00
    })

    # INT8
    p_m2, size8 = onnx_params_m_and_size_mb(int8_path)  # param count same, but size changes
    s8 = mk_sess(int8_path)
    acc8, f18 = eval_acc_f1(s8, eval_loader)
    ms8, fps8 = bench(s8)
    rows.append({
        "Group": name, "Variant": "INT8",
        "Parameters (M)": round(p_m2,3), "Disk Size (MB)": round(size8,2),
        "Inference (ms)": round(ms8,2), "Throughput (fps)": round(fps8,1),
        "Accuracy (%)": round(acc8,2), "Macro F1 (%)": round(f18,2),
        "Speed Up": round(ms32/ms8,2) if (ms8>0) else None  # speedup vs this model's FP32
    })
    return s32, s8, size32, size8

# Per-model rows + collect sessions for ensemble
fp32_sessions, int8_sessions, sizes32, sizes8 = {}, {}, {}, {}
for k,v in PATHS.items():
    s32, s8, sz32, sz8 = add_model_rows(k, v["fp32"], v["int8"])
    if k!="student":  # teachers only for ensemble
        fp32_sessions[k] = s32; int8_sessions[k] = s8; sizes32[k]=sz32; sizes8[k]=sz8

# Ensemble (teachers only)
teacher_fp32 = [fp32_sessions["mobilenet_v2"], fp32_sessions["efficientnet_b0"], fp32_sessions["squeezenet1_1"]]
teacher_int8 = [int8_sessions["mobilenet_v2"], int8_sessions["efficientnet_b0"], int8_sessions["squeezenet1_1"]]
ens_size32 = sum(sizes32.values()); ens_size8 = sum(sizes8.values())

ens_acc32, ens_f132 = ensemble_eval(teacher_fp32, eval_loader)
ens_ms32, ens_fps32 = ensemble_bench(teacher_fp32)
rows.append({
    "Group":"Ensemble (3x)","Variant":"FP32",
    "Parameters (M)": None, "Disk Size (MB)": round(ens_size32,2),
    "Inference (ms)": round(ens_ms32,2), "Throughput (fps)": round(ens_fps32,1),
    "Accuracy (%)": round(ens_acc32,2), "Macro F1 (%)": round(ens_f132,2),
    "Speed Up": 1.00
})

ens_acc8, ens_f18 = ensemble_eval(teacher_int8, eval_loader)
ens_ms8, ens_fps8 = ensemble_bench(teacher_int8)
rows.append({
    "Group":"Ensemble (3x)","Variant":"INT8",
    "Parameters (M)": None, "Disk Size (MB)": round(ens_size8,2),
    "Inference (ms)": round(ens_ms8,2), "Throughput (fps)": round(ens_fps8,1),
    "Accuracy (%)": round(ens_acc8,2), "Macro F1 (%)": round(ens_f18,2),
    "Speed Up": round(ens_ms32/ens_ms8,2) if (ens_ms8>0) else None
})

table = pd.DataFrame(rows, columns=[
    "Group","Variant","Parameters (M)","Disk Size (MB)","Inference (ms)","Throughput (fps)",
    "Accuracy (%)","Macro F1 (%)","Speed Up"
])

# Optional: order rows nicely
order = ["mobilenet_v2","efficientnet_b0","squeezenet1_1","student","Ensemble (3x)"]
table["__ord__"] = table["Group"].map({n:i for i,n in enumerate(order)})
table.sort_values(["__ord__","Variant"], inplace=True)
table.drop(columns="__ord__", inplace=True)

# Save & show
out_csv = ROOT/"onnx_fp32_int8_comprehensive.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
table.to_csv(out_csv, index=False)

print("\n=== FP32 vs INT8 — Teachers, Student, Ensemble (CPU / ONNX Runtime) ===\n")
print(table.to_string(index=False))
print(f"\nSaved CSV -> {out_csv}")


2025-08-13 06:57:17.964431301 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.17/conv/conv.1/conv.1.2/Constant_1_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:57:17.964489714 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.17/conv/conv.0/conv.0.2/Constant_1_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:57:17.964495773 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.16/conv/conv.1/conv.1.2/Constant_output_0'. It is not used by any node and should be removed from the model.
2025-08-13 06:57:17.964500821 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '/features/features.16/conv/conv.0/conv.0.2/Constant_output_0'. It is not used by any node and should be removed from the model.
2025-08-


=== FP32 vs INT8 — Teachers, Student, Ensemble (CPU / ONNX Runtime) ===

          Group Variant  Parameters (M)  Disk Size (MB)  Inference (ms)  Throughput (fps)  Accuracy (%)  Macro F1 (%)  Speed Up
   mobilenet_v2    FP32           2.211            8.47           37.27             214.7         91.40         91.51      1.00
   mobilenet_v2    INT8           2.279            2.49           29.18             274.1         83.49         83.24      1.28
efficientnet_b0    FP32           3.990           15.29          102.13              78.3         88.72         88.81      1.00
efficientnet_b0    INT8           4.112            4.66           81.10              98.6         88.17         88.43      1.26
  squeezenet1_1    FP32           0.724            2.78           21.78             367.4         81.98         82.06      1.00
  squeezenet1_1    INT8           0.736            0.80           23.98             333.6         81.84         82.02      0.91
        student    FP32       

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
